# PII benchmark: rules vs Arabic NER models (Step 2)

Runs the **same harness** as `scripts/eval_pii.py` on the **same synthetic sentences** (`eval/pii_cases.jsonl`)
for the production rules and several Arabic NER models.

**Privacy rule:** only the synthetic benchmark sentences are used here. Never paste real user questions,
API keys or the scraped laws into this notebook.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

## 1. Upload the project
On GitHub: **Code → Download ZIP**. Then run this cell and choose that zip file.

In [ ]:
from google.colab import files
import zipfile, os, sys, glob
uploaded = files.upload()                      # choose legal-online-pipeline-main.zip
name = next(iter(uploaded))
zipfile.ZipFile(name).extractall('/content/repo')
root = glob.glob('/content/repo/*/app')[0].rsplit('/app', 1)[0]
os.chdir(root); sys.path.insert(0, root)
print('project folder:', root)
print('cases file found:', os.path.exists('eval/pii_cases.jsonl'))

## 2. Install the model libraries (2–3 minutes)

In [ ]:
!pip -q install transformers gliner
!pip -q install gliner2 || echo "gliner2 not installed; that model will be skipped"
import torch; print('GPU available:', torch.cuda.is_available())

## 3. Baseline: the production rules

In [ ]:
import json, os
from app.pii.detectors import RuleDetector, HybridDetector, hf_token_classification, gliner_detector, gliner2_detector
from app.pii.pii_eval import load_cases, evaluate, summarize, format_report, misses
cases = load_cases('eval/pii_cases.jsonl')
os.makedirs('/content/results', exist_ok=True)
results = {}

def run(det):
    r = evaluate(det, cases); s = summarize(r)
    results[s['detector']] = (s, r)
    with open(f"/content/results/pii_eval_{s['detector'].replace('/', '_')}.json", 'w', encoding='utf-8') as f:
        json.dump({'summary': s, 'result': r}, f, ensure_ascii=False)
    print(format_report(s)); print('-' * 80)
    return s

rules = RuleDetector()
run(rules)

## 4. Load the candidate models
Each cell is independent: if one model fails to download or load, skip it and continue.

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else -1
models = {}
try:
    models['camelbert-da'] = hf_token_classification('CAMeL-Lab/bert-base-arabic-camelbert-da-ner',
                                                     {'pers': 'name', 'loc': 'address'}, name='camelbert-da', device=DEVICE)
except Exception as e: print('camelbert-da failed:', e)
try:
    models['camelbert-mix'] = hf_token_classification('CAMeL-Lab/bert-base-arabic-camelbert-mix-ner',
                                                      {'pers': 'name', 'loc': 'address'}, name='camelbert-mix', device=DEVICE)
except Exception as e: print('camelbert-mix failed:', e)

In [ ]:
GLINER_LABELS = {  # prompt label shown to the model -> our type (Arabic and English prompts)
    'person': 'name', 'شخص': 'name',
    'phone number': 'phone', 'رقم هاتف': 'phone',
    'national id number': 'national_id', 'الرقم الوطني': 'national_id',
    'iban': 'iban', 'email': 'email', 'بريد إلكتروني': 'email',
    'address': 'address', 'عنوان': 'address',
    'vehicle plate number': 'plate', 'رقم لوحة السيارة': 'plate',
}
try:
    models['gliner-arabic'] = gliner_detector('NAMAA-Space/gliner_arabic-v2.1', GLINER_LABELS,
                                              threshold=0.5, name='gliner-arabic')
except Exception as e: print('gliner-arabic failed:', e)
try:
    models['gliner2-arabic'] = gliner2_detector('Elafnawaf/gliner2-arabic-multitask', GLINER_LABELS, name='gliner2-arabic')
except Exception as e: print('gliner2-arabic failed (skipping):', e)
print('loaded:', list(models))

## 5. Each model on its own (all types it can detect)

In [ ]:
for m in models.values():
    run(m)

## 6. Hybrid: production rules + model, model used for names only
Structured types stay with the rules; the model only adds names.

In [ ]:
for m in models.values():
    run(HybridDetector(rules, m, model_types=('name',)))

## 7. GLiNER threshold sweep (names only, hybrid)
Lower threshold = more names caught but more false alarms.

In [ ]:
if 'gliner-arabic' in models:
    for th in (0.3, 0.4, 0.6):
        m = gliner_detector('NAMAA-Space/gliner_arabic-v2.1', GLINER_LABELS, threshold=th, name=f'gliner-arabic@{th}')
        run(HybridDetector(rules, m, model_types=('name',)))

## 8. Comparison table (copy this into the chat)

In [ ]:
print(f"{'detector':45s} {'core leak':>9s} {'names':>6s} {'min struct.':>11s} {'extra leak':>10s} {'neg FA':>7s} {'FA spans':>8s} {'ms/sent':>8s}")
for name, (s, r) in sorted(results.items(), key=lambda kv: (kv[1][0]['core']['sentence_leak_rate'], kv[1][0]['false_alarms']['rate'])):
    bt = s['core']['by_type']
    struct = min(bt[t]['recall'] for t in ('phone', 'national_id', 'iban', 'email') if t in bt)
    print(f"{name:45s} {s['core']['sentence_leak_rate']:>9.1%} {bt.get('name', {}).get('recall', 0):>6.0%} {struct:>11.0%} "
          f"{s['extra_not_in_scope']['sentence_leak_rate']:>10.1%} {s['false_alarms']['rate']:>7.0%} "
          f"{s['false_alarms']['total_false_alarm_spans']:>8d} {s['latency_ms']['median']:>8.1f}")

## 9. Errors of the best hybrid (names still missed, and false alarms)

In [ ]:
hyb = [k for k in results if k.startswith('rules+')]
if hyb:
    best = min(hyb, key=lambda k: (results[k][0]['core']['sentence_leak_rate'], results[k][0]['false_alarms']['rate']))
    s, r = results[best]; print('best hybrid:', best)
    print('\nNames not fully masked:')
    for m in misses(r):
        if m['type'] == 'name': print(' ', m['case'], m['status'], m['value'], '|', m['text'])
    print('\nFalse alarms:')
    for f in r['false_alarms']: print(' ', f['case'], f['type'], repr(f['span']), '|', f['text'])

## 10. Download all results
Send the zip back in the chat, together with the table from step 8.

In [ ]:
import shutil
shutil.make_archive('/content/pii_results', 'zip', '/content/results')
files.download('/content/pii_results.zip')